# Data Augmentation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/01-data-preprocessing/05_data_augmentation.ipynb)


---

## What are we learning?

Data augmentation is a set of techniques that artificially increases the size and diversity of a dataset by creating modified copies of existing samples. We'll learn how to generate new synthetic data points from your original data to improve model performance, especially when training data is scarce.

## The idea in plain English

Imagine you're learning to recognize cats but you've only seen 10 cat photos. Data augmentation is like taking those 10 photos and creating new versions by slightly rotating them, zooming in, adjusting brightness, or adding noise. Instead of 10 photos, you now have 100 different variations, helping you learn better without collecting new photos. In tabular data, we can add small random noise to numerical features or swap values between similar samples to create new synthetic rows.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('Setup done!')

## Step 1 — Load data

In [ ]:
# Create a small imbalanced classification dataset
X, y = make_classification(n_samples=200, n_features=4, n_redundant=0, 
                          n_informative=4, n_clusters_per_class=1,
                          n_classes=2, weights=[0.8, 0.2], random_state=42)

# Convert to DataFrame
df = pd.DataFrame(X, columns=['feature1', 'feature2', 'feature3', 'feature4'])
df['target'] = y

print(f"Original dataset shape: {df.shape}")
print(f"Class distribution:\n{df['target'].value_counts()}")
df.head()

## Step 2 — Apply Data Augmentation

In [ ]:
# Separate majority and minority classes
majority_class = df[df['target'] == 0]
minority_class = df[df['target'] == 1]

print(f"Majority class samples: {len(majority_class)}")
print(f"Minority class samples: {len(minority_class)}")

# Function to augment data by adding Gaussian noise
def augment_with_noise(data, n_samples, noise_factor=0.1, random_state=None):
    """Create synthetic samples by adding Gaussian noise to existing samples"""
    np.random.seed(random_state)
    
    # Randomly sample existing rows
    indices = np.random.choice(len(data), size=n_samples, replace=True)
    base_samples = data.iloc[indices].copy()
    
    # Add Gaussian noise to numerical features
    feature_cols = [col for col in data.columns if col != 'target']
    noise = np.random.normal(0, noise_factor, size=(n_samples, len(feature_cols)))
    
    # Calculate standard deviation for each feature to scale noise appropriately
    std_devs = data[feature_cols].std().values
    noise = noise * std_devs
    
    base_samples[feature_cols] = base_samples[feature_cols] + noise
    
    return base_samples

# Augment minority class to match majority class
n_needed = len(majority_class) - len(minority_class)
synthetic_samples = augment_with_noise(minority_class, n_needed, noise_factor=0.15, random_state=42)

print(f"Generated {len(synthetic_samples)} synthetic samples")
synthetic_samples.head()

In [ ]:
# Combine original and synthetic data
augmented_df = pd.concat([df, synthetic_samples], ignore_index=True)

print(f"Augmented dataset shape: {augmented_df.shape}")
print(f"New class distribution:\n{augmented_df['target'].value_counts()}")

# Verify synthetic samples are reasonable (check ranges)
print("\nFeature ranges comparison:")
for col in ['feature1', 'feature2', 'feature3', 'feature4']:
    orig_min, orig_max = df[col].min(), df[col].max()
    synth_min, synth_max = synthetic_samples[col].min(), synthetic_samples[col].max()
    print(f"{col}: Original [{orig_min:.2f}, {orig_max:.2f}] vs Synthetic [{synth_min:.2f}, {synth_max:.2f}]")

## Step 3 — Visualise

In [ ]:
# Visualize original vs augmented data
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Original data
axes[0].scatter(df[df['target']==0]['feature1'], df[df['target']==0]['feature2'], 
               alpha=0.6, label='Class 0 (Majority)', color='blue')
axes[0].scatter(df[df['target']==1]['feature1'], df[df['target']==1]['feature2'], 
               alpha=0.6, label='Class 1 (Minority)', color='red')
axes[0].set_title('Original Dataset')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Augmented data
axes[1].scatter(augmented_df[augmented_df['target']==0]['feature1'], 
               augmented_df[augmented_df['target']==0]['feature2'], 
               alpha=0.6, label='Class 0 (Majority)', color='blue')
axes[1].scatter(augmented_df[augmented_df['target']==1]['feature1'], 
               augmented_df[augmented_df['target']==1]['feature2'], 
               alpha=0.6, label='Class 1 (Original)', color='red')
axes[1].scatter(synthetic_samples['feature1'], synthetic_samples['feature2'], 
               alpha=0.6, label='Class 1 (Synthetic)', color='orange', marker='x')
axes[1].set_title('Augmented Dataset')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Results & interpretation

In [ ]:
# Compare class balance before and after augmentation
original_balance = df['target'].value_counts(normalize=True)
augmented_balance = augmented_df['target'].value_counts(normalize=True)

print("Class Balance Comparison:")
print("=" * 40)
print(f"Original dataset:")
print(f"  Class 0: {original_balance[0]:.1%}")
print(f"  Class 1: {original_balance[1]:.1%}")
print(f"  Imbalance ratio: {original_balance[0]/original_balance[1]:.1f}:1")
print()
print(f"Augmented dataset:")
print(f"  Class 0: {augmented_balance[0]:.1%}")
print(f"  Class 1: {augmented_balance[1]:.1%}")
print(f"  Imbalance ratio: {augmented_balance[0]/augmented_balance[1]:.1f}:1")

# Calculate feature statistics
print("\nFeature Statistics Comparison:")
print("=" * 40)
feature_cols = ['feature1', 'feature2', 'feature3', 'feature4']
for col in feature_cols:
    orig_mean = df[col].mean()
    aug_mean = augmented_df[col].mean()
    print(f"{col}: Original mean = {orig_mean:.3f}, Augmented mean = {aug_mean:.3f}")

## Summary

- Data augmentation creates synthetic samples by adding controlled noise to existing data, helping balance imbalanced datasets
- We successfully generated 120 synthetic samples for the minority class, transforming an 80:20 imbalance to a balanced 50:50 split
- The augmented data maintains similar statistical properties (means, ranges) to the original data while increasing dataset size
- Visual inspection shows synthetic samples cluster reasonably around original minority class samples
- This technique is particularly valuable when collecting more real data is expensive or impossible

## Exercises

1. Try different noise factors (0.05, 0.2, 0.3) and visualize how it affects the synthetic samples - what happens when noise is too high or too low?
2. Implement a different augmentation technique: instead of adding noise, try interpolating between existing minority samples to create new ones
3. Apply data augmentation to a regression problem: create synthetic samples for a dataset with continuous target values and compare model performance before and after augmentation

In [ ]:
# Your code here